In [3]:
import torch
import torch.nn as nn

## EfficientNet

In [4]:
class SEBlock(nn.Module):
    def __init__(self, channels, reduction):
        super().__init__()

        hidden = channels // reduction

        self.pool = nn.AdaptiveAvgPool2d(1)

        self.layer = nn.Sequential(
            nn.Conv2d(channels, hidden, 1),
            nn.SiLU(),

            nn.Conv2d(hidden, channels, 1),
            nn.Sigmoid()
        )
        
    def forward(self, X):
        scale = self.pool(X)
        scale = self.layer(scale)

        return X * scale
        

In [7]:
class MBConv(nn.Module):
    def __init__(self, in_channels, out_channels, expand_ratio, stride= 1):
        super().__init__()

        hidden_layer = in_channels * expand_ratio

        self.expand = nn.Sequential(
            nn.Conv2d(in_channels, hidden_layer, 1, bias= False),
            nn.BatchNorm2d(hidden_layer),
            nn.SiLU()
        )

        self.depthwise = nn.Sequential(
            nn.Conv2d(hidden_layer, hidden_layer, 3, stride= stride, padding= 1, bias= False, groups= hidden_layer),
            nn.BatchNorm2d(hidden_layer),
            nn.SiLU(),
        )

        self.se = SEBlock(hidden_layer, 4)

        self.project = nn.Sequential(
            nn.Conv2d(hidden_layer, out_channels, 1, bias= False),
            nn.BatchNorm2d(out_channels)
        )

        self.useResidual = (stride == 1 and in_channels == out_channels)

    def forward(self, x):

        identity = x

        x = self.expand(x)
        x = self.depthwise(x)
        x = self.se(x)
        x = self.project(x)

        if self.use_residual:
            x = x + identity

        return x